In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:csv_path = os.path.join(path, "updated_pollution_dataset.csv")
csv_path = os.path.join(path, "Q1_data.csv")

data = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
data.head()

In [ ]:
# Task 3: Write your code here:
data.info()

In [ ]:
# Task 4: Write your code here:
data.describe()

In [ ]:
# Task 5: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(data['Delivery_Time'].dropna(), bins=30, edgecolor='black', color='orange')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
data = df.drop('Order_ID', axis=1)
data.info()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(data):

  # Get missing values using pandas
  missing_values = data.isnull().sum()

  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(data)

data = data.fillna(data.mean())
data.info()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(data)

In [ ]:
data.info()

In [ ]:
def encode_categorical_columns(df):
    categorical_cols = df.select_dtypes(include=["object"]).columns
    print("Categorical Columns:", list(categorical_cols))

label_encoders = encode_categorical_columns(data)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

data.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
numerical_cols = data.select_dtypes(include=["number"]).columns.drop("Delivery_Time")
scaler = StandardScaler()

# scale the `numerical_cols`
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

data.head()

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(data, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = data.drop("Delivery_Time",axis=1)
y = data['Delivery_Time']

In [ ]:
def mean_absolute_error(y, y_hat):
  return  (1 / (2 * len(y))) * np.sum((y_hat - y))

In [ ]:
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_absolute_error(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.metrics import mean_absolute_error as sklearn_mae

In [ ]:
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
lr_losses = []
lr_mse = []

lr_r2 = []
lr_mae = []

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mse = sklearn_mae(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_losses.append(losses)
  lr_r2.append(r2)

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error as mae,  r2_score
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
lr_losses = []
lr_mae = []
lr_r2 = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test, y_pred)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_losses.append(losses)
  lr_mae.append(mae)
  #lr_rmse.append(rmse)
  #lr_r2.append(r2)

In [ ]:
# Task 1: Write your code here:
average_losses = np.mean(lr_losses, axis=0)
plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(data["Delivery_Time"].dropna(), bins=30, edgecolor='black', color='orange')
plt.title('dilivery time distribution in histograph ')
plt.xlabel('Year')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: